# 02b — Probability & Inference Analyst: Inferensi Statistik (Tahap 3, $\alpha = 0.05$)

**Peran:** Probability & Inference Analyst *(peluang bersyarat, distribusi teoretis, selang kepercayaan, dan uji hipotesis)*  
**Dikerjakan oleh:** Rafli Adriana Saputra (257006111004)  
**Kelompok / Tema:** Kelompok 12 — Tema B (Kualitas Udara & Mobilitas Perkotaan)  

### Ruang Lingkup Analisis Tahap 3 (Minggu 3)
Sampel penelitian ($N$ observasi bersih) digunakan untuk menarik kesimpulan (*inferensi*) mengenai parameter populasi kualitas udara perkotaan pada taraf signifikansi $\alpha = 0,05$:

| Bagian | Fokus Analisis & Pertanyaan Riset Tema B | Metode Statistik | Acuan Panduan |
| :--- | :--- | :--- | :--- |
| **A** | Estimasi kisaran rata-rata `Indeks_PM25` populasi dan proporsi kategori `Status_Udara = Tidak Sehat` | Selang Kepercayaan (Confidence Interval 95%) | Checklist 3a |
| **B** | Apakah rata-rata `Indeks_PM25` berbeda signifikan antara **Hari Kerja** dan **Akhir Pekan**? | Two-Sample Independent t-Test | Checklist 3b-1 |
| **C** | Apakah terdapat perbedaan rata-rata `Indeks_PM25` di ketiga **Zona_Lokasi**? | One-Way ANOVA (+ Post-Hoc Tukey HSD) | Checklist 3b-2 |
| **D** | Apakah `Status_Udara` berkaitan dengan `Jam_Sibuk`? | Chi-Square Test of Independence (Observasi vs Ekspektasi) | Checklist 3b-3 |

> **Kaidah Keputusan Hipotesis ($\alpha = 0,05$):** Jika $p\text{-value} < 0,05$, maka **$H_0$ ditolak** (terdapat perbedaan atau hubungan yang signifikan secara statistik). Jika $p\text{-value} \ge 0,05$, maka **$H_0$ gagal ditolak** (belum cukup bukti statistik untuk menyatakan adanya perbedaan atau hubungan).

In [ ]:
# ============ SETUP (jalankan sel ini PERTAMA) ============
import os, sys, subprocess
GITHUB_USER = "Tur-ai"                    # <- username GitHub pemilik repo
GITHUB_REPO = "udara-mobilitas-statsprob" # <- nama repository
BRANCH = "main"

if "google.colab" in sys.modules:
    PROYEK = "/content/proyek"
    if not os.path.exists(PROYEK):
        token = ""
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN") + "@"
        except Exception:
            pass
        url = f"https://{token}github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
        r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, PROYEK],
                           capture_output=True, text=True)
        if r.returncode != 0:
            pesan = (r.stderr or "").replace(token.rstrip("@"), "***") if token else (r.stderr or "")
            raise RuntimeError("Gagal clone repo. Cek GITHUB_USER/GITHUB_REPO, atau lihat bagian "
                               "'Troubleshooting' di docs/02_panduan_github_colab.md\n" + pesan)
    else:
        subprocess.run(["git", "-C", PROYEK, "pull"], capture_output=True)
    os.chdir(PROYEK)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from IPython.display import display, Markdown
from src.helpers import *

sns.set_theme(style="whitegrid", font_scale=1.0)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
print("Setup selesai. Folder kerja:", os.getcwd())

In [ ]:
df = muat_data_bersih()
N = len(df)
VAR = "Indeks_PM25"
ALPHA = globals().get("ALPHA", 0.05)
hasil_uji = {}      # tempat merangkum hasil semua uji

print(f"Jumlah observasi (N) : {N}")
print(f"Taraf signifikansi   : {ALPHA}")
df.head()

## A. Selang Kepercayaan / Confidence Interval 95% (Tahap 3a)

Selang kepercayaan (*Confidence Interval* 95%) memberikan rentang estimasi interval bagi parameter populasi sebenarnya berdasarkan data sampel dengan tingkat keyakinan 95%.

### A1. CI 95% Rata-rata Populasi `Indeks_PM25`
$$\bar{x} \pm t_{\alpha/2,\,n-1} \cdot \frac{s}{\sqrt{n}}$$

In [ ]:
x = df[VAR].dropna()
n = len(x)
xbar, s = x.mean(), x.std(ddof=1)
se = s / np.sqrt(n)                              # standard error
t_krit = stats.t.ppf(1 - ALPHA / 2, df=n - 1)    # nilai t kritis
bb, ba = xbar - t_krit * se, xbar + t_krit * se
print(f"n = {n} | mean = {xbar:.3f} | s = {s:.3f} | SE = {se:.3f} | t kritis = {t_krit:.3f}")
print(f"CI 95% rata-rata PM2.5 = [{bb:.2f} ; {ba:.2f}]")

# Cek dengan fungsi scipy (harus sama)
cek = stats.t.interval(0.95, df=n - 1, loc=xbar, scale=se)
print("Cek scipy:", np.round(cek, 2))
hasil_uji["CI_mean"] = (xbar, bb, ba)

### A2. CI 95% Proporsi Populasi Kategori `Status_Udara = "Tidak Sehat"`
$$\hat{p} \pm z_{\alpha/2}\sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$
Perhitungan menggunakan pendekatan normal (Wald, valid apabila $n\hat{p} \ge 5$ dan $n(1-\hat{p}) \ge 5$) serta dibandingkan dengan metode Wilson sebagai pembanding ketahanan.

In [ ]:
kasus = int((df["Status_Udara"].astype(str).str.strip() == "Tidak Sehat").sum())
p_hat = kasus / N
z_krit = stats.norm.ppf(1 - ALPHA / 2)
se_p = np.sqrt(p_hat * (1 - p_hat) / N)
print(f"p topi = {kasus}/{N} = {p_hat:.4f}")
print(f"Syarat: n*p = {N*p_hat:.1f} (>=5?)  n*(1-p) = {N*(1-p_hat):.1f} (>=5?)")
wald = (p_hat - z_krit * se_p, p_hat + z_krit * se_p)
wilson = proportion_confint(kasus, N, alpha=ALPHA, method="wilson")
print(f"CI 95% Wald   : [{wald[0]:.4f} ; {wald[1]:.4f}]  -> [{wald[0]*100:.2f}% ; {wald[1]*100:.2f}%]")
print(f"CI 95% Wilson : [{wilson[0]:.4f} ; {wilson[1]:.4f}]  -> [{wilson[0]*100:.2f}% ; {wilson[1]*100:.2f}%]")

tabel_ci = pd.DataFrame([
    ["Rata-rata PM2.5", round(xbar, 3), round(bb, 3), round(ba, 3), "t-interval"],
    ["Proporsi Tidak Sehat", round(p_hat, 4), round(wald[0], 4), round(wald[1], 4), "Wald (normal)"],
    ["Proporsi Tidak Sehat", round(p_hat, 4), round(wilson[0], 4), round(wilson[1], 4), "Wilson (pembanding)"],
], columns=["Parameter", "Estimasi", "Batas_Bawah", "Batas_Atas", "Metode"])
simpan_tabel(tabel_ci, "03_confidence_interval", index=False)
tabel_ci

In [ ]:
# Render otomatis interpretasi CI 95% dalam bahasa sehari-hari
status_ambang = (
    "masih berada di bawah ambang batas kategori Tidak Sehat (65), yang berarti secara rata-rata kualitas udara kota berada pada kategori Sedang meskipun tetap memiliki lonjakan polusi pada waktu atau zona tertentu"
    if ba < 65 else
    "telah menyentuh atau melampaui ambang batas kategori Tidak Sehat (65), yang menandakan paparan polusi rata-rata yang tinggi di wilayah perkotaan"
)
teks_interpretasi_ci = (
    f"Dengan tingkat kepercayaan 95%, rata-rata Indeks PM2.5 populasi diperkirakan berada di antara **{bb:.2f}** hingga **{ba:.2f}** (estimasi titik = **{xbar:.2f}**). "
    f"Batas atas selang kepercayaan ({ba:.2f}) {status_ambang}. "
    f"Sementara itu, proporsi waktu/pengukuran dengan status udara **Tidak Sehat** di populasi diperkirakan berada antara **{wald[0]*100:.2f}%** hingga **{wald[1]*100:.2f}%** (metode Wald; atau **{wilson[0]*100:.2f}%** hingga **{wilson[1]*100:.2f}%** dengan metode Wilson)."
)
display(Markdown(f"### Interpretasi Selang Kepercayaan (CI 95%)\n{teks_interpretasi_ci}"))

## B. Two-Sample Independent t-Test (Tahap 3b-1)

**Pertanyaan Riset Tema B:** Apakah rata-rata `Indeks_PM25` berbeda signifikan antara **Hari Kerja** dan **Akhir Pekan**?

* **$H_0$:** $\mu_{\text{Hari Kerja}} = \mu_{\text{Akhir Pekan}}$ *(tidak terdapat perbedaan rata-rata `Indeks_PM25` antara hari kerja dan akhir pekan)*
* **$H_1$:** $\mu_{\text{Hari Kerja}} \neq \mu_{\text{Akhir Pekan}}$ *(terdapat perbedaan rata-rata `Indeks_PM25` yang signifikan antara hari kerja dan akhir pekan — uji dua arah)*
* **Taraf Signifikansi:** $\alpha = 0,05$

In [ ]:
g1 = df.loc[df["Jenis_Hari"].astype(str).str.strip() == "Hari Kerja", VAR].dropna()
g2 = df.loc[df["Jenis_Hari"].astype(str).str.strip() == "Akhir Pekan", VAR].dropna()
desk = pd.DataFrame({
    "n": [len(g1), len(g2)],
    "Mean": [g1.mean(), g2.mean()],
    "Std": [g1.std(ddof=1), g2.std(ddof=1)]
}, index=["Hari Kerja", "Akhir Pekan"]).round(2)
desk

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=df, x="Jenis_Hari", y=VAR, hue="Jenis_Hari", order=["Hari Kerja", "Akhir Pekan"],
            palette={"Hari Kerja": "#e07b39", "Akhir Pekan": "#4a90c2"}, legend=False, ax=axes[0])
axes[0].set_title("Indeks PM2.5 menurut Jenis Hari"); axes[0].set_xlabel("")
sns.histplot(data=df, x=VAR, hue="Jenis_Hari", bins=25, stat="density", common_norm=False, alpha=0.5, ax=axes[1])
axes[1].set_title("Sebaran Indeks PM2.5 per Kelompok Hari")
plt.tight_layout()
simpan_gambar(fig, "03_ttest_boxplot")
plt.show()

### Pemeriksaan Asumsi Sebelum Uji-t
1. **Independensi Sampel:** Pengamatan pada Hari Kerja dan Akhir Pekan berasal dari catatan waktu yang terpisah dan saling bebas.
2. **Normalitas:** Dengan ukuran sampel yang memadai ($n > 30$ per kelompok), Teorema Limit Pusat menjamin ketahanan (*robustness*) uji-t terhadap penyimpangan ringan dari normalitas.
3. **Homogenitas Varians (Uji Levene):** Jika $p_{\text{Levene}} \ge 0,05$, digunakan *Student's t-test* (varians homogen); jika $p_{\text{Levene}} < 0,05$, digunakan *Welch's t-test* (varians heterogen).

In [ ]:
lev_stat, lev_p = stats.levene(g1, g2)
sama = lev_p >= ALPHA
print(f"Levene: stat = {lev_stat:.3f}, p = {lev_p:.4f} -> varians {'SAMA' if sama else 'BERBEDA'}")

t_stat, p_t = stats.ttest_ind(g1, g2, equal_var=sama)
if sama:
    df_t = len(g1) + len(g2) - 2
else:
    v1, v2 = g1.var(ddof=1), g2.var(ddof=1); n1, n2 = len(g1), len(g2)
    df_t = (v1/n1 + v2/n2)**2 / ((v1/n1)**2/(n1-1) + (v2/n2)**2/(n2-1))

selisih = g1.mean() - g2.mean()
sp = np.sqrt(((len(g1)-1)*g1.var(ddof=1) + (len(g2)-1)*g2.var(ddof=1)) / (len(g1)+len(g2)-2))
cohen_d = selisih / sp
keputusan_t = "Tolak H0" if p_t < ALPHA else "Gagal tolak H0"

print(f"Metode          : {'Student (varians sama)' if sama else 'Welch (varians beda)'}")
print(f"t = {t_stat:.3f}, df = {df_t:.1f}, p-value = {p_t:.4g}")
print(f"Selisih rata-rata (Hari Kerja - Akhir Pekan) = {selisih:.2f}")
print(f"Cohen's d = {cohen_d:.2f}  (0.2 kecil, 0.5 sedang, 0.8 besar)")
print("KEPUTUSAN:", "TOLAK H0 (p < 0.05)" if p_t < ALPHA else "GAGAL TOLAK H0 (p >= 0.05)")

tabel_t = pd.DataFrame([{
    "Uji": "t independen " + ("(Student)" if sama else "(Welch)"),
    "Mean_Hari_Kerja": round(g1.mean(), 2), "Mean_Akhir_Pekan": round(g2.mean(), 2),
    "Selisih": round(selisih, 2), "t": round(t_stat, 3), "df": round(df_t, 1),
    "p_value": p_t, "Cohen_d": round(cohen_d, 2), "Levene_p": round(lev_p, 4),
    "Keputusan": keputusan_t}])
simpan_tabel(tabel_t, "03_ttest_hasil", index=False)

In [ ]:
# Cek ketahanan: uji non-parametrik Mann-Whitney U (pembanding tanpa asumsi normalitas)
u, p_u = stats.mannwhitneyu(g1, g2, alternative="two-sided")
print(f"Mann-Whitney U: U = {u:.0f}, p = {p_u:.4g}  (pembanding saja; yang dilaporkan utama tetap uji-t)")

In [ ]:
# Render lengkap hasil Two-Sample Independent t-Test sesuai Checklist Minggu 3
arah_t = "lebih tinggi" if selisih > 0 else "lebih rendah"
if p_t < ALPHA:
    interp_t = (
        f"Rata-rata Indeks PM2.5 pada **Hari Kerja ({g1.mean():.2f})** secara statistik terbukti **{arah_t} dan berbeda nyata** "
        f"dibandingkan **Akhir Pekan ({g2.mean():.2f})**, dengan selisih sebesar **{abs(selisih):.2f} poin** ($t = {t_stat:.3f}$, $p = {p_t:.4g} < 0,05$, Cohen's $d = {cohen_d:.2f}$). "
        f"Artinya, intensitas mobilitas dan aktivitas harian pada hari kerja memberikan dampak nyata terhadap peningkatan konsentrasi partikulat PM2.5 di udara."
    )
else:
    interp_t = (
        f"Rata-rata Indeks PM2.5 pada **Hari Kerja ({g1.mean():.2f})** **tidak berbeda secara signifikan** dibandingkan **Akhir Pekan ({g2.mean():.2f})**, "
        f"dengan selisih hanya **{abs(selisih):.2f} poin** ($t = {t_stat:.3f}$, $p = {p_t:.4g} \ge 0,05$). "
        f"Artinya, tingkat polusi PM2.5 cenderung konsisten antara hari kerja dan akhir pekan sehingga faktor spasial (zona) atau kondisi meteorologis perlu dianalisis lebih lanjut."
    )

display(Markdown(f"""
### Ringkasan Hasil Two-Sample Independent t-Test (Checklist 3b-1)

| Komponen | Hasil Analisis |
| :--- | :--- |
| **$H_0$** | $\mu_{{\text{{Hari Kerja}}}} = \mu_{{\text{{Akhir Pekan}}}}$ (Tidak ada perbedaan rata-rata `Indeks_PM25` antara Hari Kerja dan Akhir Pekan) |
| **$H_1$** | $\mu_{{\text{{Hari Kerja}}}} \neq \mu_{{\text{{Akhir Pekan}}}}$ (Terdapat perbedaan rata-rata `Indeks_PM25` antara Hari Kerja dan Akhir Pekan) |
| **Statistik Uji** | $t = {t_stat:.3f}$ ($df = {df_t:.1f}$, metode {'Student' if sama else 'Welch'}) |
| **$p$-value** | ${p_t:.4g}$ |
| **Keputusan ($\alpha = 0,05$)** | **{keputusan_t}** |
| **Interpretasi Bahasa Sehari-hari** | {interp_t} |
"""))

## C. One-Way ANOVA dan Post-Hoc Tukey HSD (Tahap 3b-2)

**Pertanyaan Riset Tema B:** Apakah terdapat perbedaan rata-rata `Indeks_PM25` di ketiga **Zona_Lokasi** (`Pusat Kota`, `Kawasan Kampus`, `Pinggiran`)?

Pengujian serentak menggunakan *One-Way ANOVA* dipilih dibandingkan melakukan uji-t berpasangan berkali-kali guna menjaga tingkat kesalahan Tipe I tetap terkendali pada $\alpha = 0,05$.

* **$H_0$:** $\mu_{\text{Pusat Kota}} = \mu_{\text{Kawasan Kampus}} = \mu_{\text{Pinggiran}}$ *(rata-rata `Indeks_PM25` sama di ketiga zona lokasi)*
* **$H_1$:** Minimal terdapat satu pasangan `Zona_Lokasi` yang memiliki rata-rata `Indeks_PM25` berbeda secara signifikan
* **Taraf Signifikansi:** $\alpha = 0,05$

In [ ]:
kel = [df.loc[df["Zona_Lokasi"].astype(str).str.strip() == z, VAR].dropna() for z in URUTAN_ZONA]
desk_z = df.groupby("Zona_Lokasi")[VAR].agg(["count", "mean", "std"]).loc[URUTAN_ZONA].round(2)
display(desk_z)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.boxplot(data=df, x="Zona_Lokasi", y=VAR, hue="Zona_Lokasi", order=URUTAN_ZONA, palette=WARNA_ZONA, legend=False, ax=ax)
ax.set_title("Distribusi Indeks PM2.5 menurut Zona Lokasi"); ax.set_xlabel("")
plt.tight_layout()
simpan_gambar(fig, "03_anova_boxplot")
plt.show()

In [ ]:
# Asumsi homogenitas varians antar zona (Uji Levene)
lev_s, lev_pz = stats.levene(*kel)
print(f"Levene antar zona: stat = {lev_s:.3f}, p = {lev_pz:.4f} -> varians {'SAMA (ANOVA aman)' if lev_pz >= ALPHA else 'BERBEDA (cek juga Kruskal-Wallis)'}")

# One-Way ANOVA
F, p_f = stats.f_oneway(*kel)
k_grup, n_tot = len(kel), sum(len(g) for g in kel)
df_antar, df_dalam = k_grup - 1, n_tot - k_grup

# Ukuran efek: eta kuadrat = SS_antar / SS_total
grand = df[VAR].dropna().mean()
ss_antar = sum(len(g) * (g.mean() - grand) ** 2 for g in kel)
ss_total = ((df[VAR].dropna() - grand) ** 2).sum()
eta2 = ss_antar / ss_total
keputusan_anova = "Tolak H0" if p_f < ALPHA else "Gagal tolak H0"

print(f"F({df_antar}, {df_dalam}) = {F:.3f}, p-value = {p_f:.4g}")
print(f"Eta kuadrat = {eta2:.3f}  -> zona menjelaskan ~{eta2*100:.1f}% variasi PM2.5  (0.01 kecil, 0.06 sedang, 0.14 besar)")
print("KEPUTUSAN:", "TOLAK H0 -> ada perbedaan -> lanjut Post-Hoc Tukey" if p_f < ALPHA else "GAGAL TOLAK H0 -> tidak perlu uji lanjut")

tabel_anova = pd.DataFrame([
    {"Sumber": "Antar zona", "df": df_antar, "SS": round(ss_antar, 2), "MS": round(ss_antar/df_antar, 2), "F": round(F, 3), "p_value": p_f},
    {"Sumber": "Dalam zona", "df": df_dalam, "SS": round(ss_total-ss_antar, 2), "MS": round((ss_total-ss_antar)/df_dalam, 2), "F": np.nan, "p_value": np.nan},
    {"Sumber": "Total", "df": n_tot-1, "SS": round(ss_total, 2), "MS": np.nan, "F": np.nan, "p_value": np.nan}
])
simpan_tabel(tabel_anova, "03_anova_hasil", index=False)
tabel_anova

### Uji Lanjut Post-Hoc Tukey HSD
Sesuai panduan analisis, apabila hasil *One-Way ANOVA* signifikan ($p < 0,05$), dilakukan uji lanjut **Post-Hoc Tukey HSD** untuk mengidentifikasi pasangan `Zona_Lokasi` mana saja yang memiliki perbedaan rata-rata `Indeks_PM25` secara nyata (`reject = True`).

In [ ]:
df_anova = df[["Zona_Lokasi", VAR]].dropna()
tukey = pairwise_tukeyhsd(endog=df_anova[VAR], groups=df_anova["Zona_Lokasi"], alpha=ALPHA)
tabel_tukey = pd.DataFrame(tukey._results_table.data[1:], columns=tukey._results_table.data[0])

if p_f < ALPHA:
    print("Hasil ANOVA signifikan (p < 0.05) -> Menampilkan uji lanjut Post-Hoc Tukey HSD:")
    print(tukey)
    simpan_tabel(tabel_tukey, "03_tukey_hasil", index=False)
else:
    print("Hasil ANOVA tidak signifikan (p >= 0.05) -> Uji lanjut Post-Hoc Tukey tidak diperlukan secara formal.")
    print(tukey)
    simpan_tabel(tabel_tukey, "03_tukey_hasil", index=False)

In [ ]:
# Pembanding ketahanan non-parametrik (Kruskal-Wallis)
H, p_kw = stats.kruskal(*kel)
print(f"Kruskal-Wallis: H = {H:.2f}, p = {p_kw:.4g}  (pembanding ketahanan distribusi)")

In [ ]:
# Render lengkap hasil One-Way ANOVA & Post-Hoc Tukey HSD sesuai Checklist Minggu 3
zona_tertinggi = desk_z["mean"].idxmax()
zona_terendah = desk_z["mean"].idxmin()
sig_pairs = tabel_tukey[tabel_tukey["reject"] == True]

if p_f < ALPHA and len(sig_pairs) > 0:
    daftar_tukey = "; ".join([
        f"{r['group1']} vs {r['group2']} (selisih = {abs(float(r['meandiff'])):.2f} poin, p-adj = {float(r['p-adj']):.4g})"
        for _, r in sig_pairs.iterrows()
    ])
    interp_anova = (
        f"Terdapat perbedaan rata-rata Indeks PM2.5 yang sangat nyata antar-zona lokasi ($F({df_antar}, {df_dalam}) = {F:.3f}$, $p = {p_f:.4g} < 0,05$, $\eta^2 = {eta2:.3f}$). "
        f"Zona dengan tingkat polusi tertinggi adalah **{zona_tertinggi}** (rata-rata **{desk_z.loc[zona_tertinggi, 'mean']:.2f}**), sedangkan zona paling bersih adalah **{zona_terendah}** (rata-rata **{desk_z.loc[zona_terendah, 'mean']:.2f}**). "
        f"Uji lanjut Tukey HSD menunjukkan pasangan zona yang berbeda signifikan meliputi: {daftar_tukey}."
    )
else:
    daftar_tukey = "Tidak ada pasangan zona yang berbeda signifikan (hasil ANOVA tidak signifikan)"
    interp_anova = (
        f"Rata-rata Indeks PM2.5 tidak berbeda secara signifikan di ketiga zona lokasi ($F({df_antar}, {df_dalam}) = {F:.3f}$, $p = {p_f:.4g} \ge 0,05$). "
        f"Meskipun rata-rata sampel tertinggi tercatat pada **{zona_tertinggi}** ({desk_z.loc[zona_tertinggi, 'mean']:.2f}) dan terendah pada **{zona_terendah}** ({desk_z.loc[zona_terendah, 'mean']:.2f}), selisih tersebut belum cukup kuat secara statistik."
    )

display(Markdown(f"""
### Ringkasan Hasil One-Way ANOVA & Post-Hoc Tukey (Checklist 3b-2)

| Komponen | Hasil Analisis |
| :--- | :--- |
| **$H_0$** | $\mu_{{\text{{Pusat Kota}}}} = \mu_{{\text{{Kawasan Kampus}}}} = \mu_{{\text{{Pinggiran}}}}$ (Rata-rata `Indeks_PM25` sama di ketiga zona lokasi) |
| **$H_1$** | Minimal terdapat satu `Zona_Lokasi` yang memiliki rata-rata `Indeks_PM25` berbeda signifikan |
| **Statistik Uji** | $F({df_antar}, {df_dalam}) = {F:.3f}$ ($\eta^2 = {eta2:.3f}$) |
| **$p$-value** | ${p_f:.4g}$ |
| **Keputusan ($\alpha = 0,05$)** | **{keputusan_anova}** |
| **Post-Hoc Tukey HSD** | {daftar_tukey} |
| **Interpretasi Bahasa Sehari-hari** | {interp_anova} |
"""))

## D. Chi-Square Test of Independence (Tahap 3b-3)

**Pertanyaan Riset Tema B:** Apakah `Status_Udara` berkaitan dengan `Jam_Sibuk`?

* **$H_0$:** `Status_Udara` **independen** (tidak berkaitan) dengan `Jam_Sibuk`
* **$H_1$:** `Status_Udara` **dependen** (berkaitan) dengan `Jam_Sibuk`
* **Taraf Signifikansi:** $\alpha = 0,05$

Statistik uji membandingkan **frekuensi observasi ($O$)** pada tabel kontingensi dengan **frekuensi ekspektasi ($E$)** apabila kedua variabel tidak saling berhubungan:

$$E_{ij} = \frac{(\text{total baris } i)(\text{total kolom } j)}{N}, \qquad \chi^2 = \sum \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$

In [ ]:
observasi = pd.crosstab(df["Status_Udara"].astype(str).str.strip(), df["Jam_Sibuk"].astype(str).str.strip())
# Urutkan baris sesuai URUTAN_STATUS jika seluruh kategorinya cocok
if set(URUTAN_STATUS).issubset(set(observasi.index)):
    observasi = observasi.reindex(URUTAN_STATUS)

chi2, p_chi, dof, ekspektasi_arr = stats.chi2_contingency(observasi)
ekspektasi = pd.DataFrame(ekspektasi_arr, index=observasi.index, columns=observasi.columns)

print("=== Tabel Kontingensi Frekuensi OBSERVASI (O) ===")
display(observasi)
print("=== Tabel Kontingensi Frekuensi EKSPEKTASI (E) jika H0 Benar ===")
display(ekspektasi.round(2))
print("Syarat validitas (semua ekspektasi >= 5):", bool((ekspektasi >= 5).all().all()),
      "| nilai sel terkecil =", round(ekspektasi.values.min(), 2))

In [ ]:
n_chi = observasi.values.sum()
cramers_v = np.sqrt(chi2 / (n_chi * (min(observasi.shape) - 1)))
keputusan_chi = "Tolak H0" if p_chi < ALPHA else "Gagal tolak H0"

print(f"Chi-square (chi2) = {chi2:.3f}, df = {dof}, p-value = {p_chi:.4g}")
print(f"Cramer's V        = {cramers_v:.3f}  (0.1 lemah, 0.3 sedang, 0.5 kuat)")
print("KEPUTUSAN:", "TOLAK H0 -> ada hubungan" if p_chi < ALPHA else "GAGAL TOLAK H0 -> tidak ada bukti hubungan")

simpan_tabel(observasi, "03_chi2_observasi")
simpan_tabel(ekspektasi.round(2), "03_chi2_ekspektasi")
tabel_chi = pd.DataFrame([{
    "Uji": "Chi-Square independensi (Status_Udara x Jam_Sibuk)",
    "chi2": round(chi2, 3), "df": dof,
    "p_value": p_chi, "Cramers_V": round(cramers_v, 3),
    "Keputusan": keputusan_chi
}])
simpan_tabel(tabel_chi, "03_chi2_hasil", index=False)

# Visualisasi perbandingan frekuensi Observasi vs Ekspektasi
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
observasi.plot(kind="bar", ax=axes[0], rot=0, color=["#4a90c2", "#e07b39"])
axes[0].set_title("Frekuensi Observasi (Data Aktual)")
axes[0].set_ylabel("Jumlah Pengamatan")
axes[0].set_xlabel("Status Udara")

ekspektasi.plot(kind="bar", ax=axes[1], rot=0, color=["#8fbce6", "#f2b385"])
axes[1].set_title("Frekuensi Ekspektasi (Jika H0 Independen)")
axes[1].set_ylabel("Jumlah Pengamatan Ekspektasi")
axes[1].set_xlabel("Status Udara")

plt.tight_layout()
simpan_gambar(fig, "03_chi2_observasi_vs_ekspektasi")
plt.show()

# Persentase per kolom Jam_Sibuk untuk interpretasi
persen = (observasi / observasi.sum() * 100).round(1)
print("Persentase Status Udara per Kategori Jam_Sibuk (%):")
persen

In [ ]:
# Render lengkap hasil Chi-Square Test of Independence sesuai Checklist Minggu 3
if p_chi < ALPHA:
    interp_chi = (
        f"Terdapat hubungan yang signifikan secara statistik antara `Status_Udara` dan `Jam_Sibuk` ($\chi^2 = {chi2:.3f}$, $df = {dof}$, $p = {p_chi:.4g} < 0,05$, Cramér's $V = {cramers_v:.3f}$). "
        f"Komposisi kategori kualitas udara berubah secara nyata ketika memasuki periode jam sibuk lalu lintas dibandingkan di luar jam sibuk."
    )
else:
    interp_chi = (
        f"Tidak ditemukan hubungan yang signifikan secara statistik antara `Status_Udara` dan `Jam_Sibuk` ($\chi^2 = {chi2:.3f}$, $df = {dof}$, $p = {p_chi:.4g} \ge 0,05$, Cramér's $V = {cramers_v:.3f}$). "
        f"Proporsi kategori status udara pada saat jam sibuk maupun di luar jam sibuk relatif serupa (frekuensi observasi mendekati frekuensi ekspektasi), sehingga status kategori udara tidak hanya ditentukan oleh jam sibuk saja."
    )

display(Markdown(f"""
### Ringkasan Hasil Chi-Square Test of Independence (Checklist 3b-3)

| Komponen | Hasil Analisis |
| :--- | :--- |
| **$H_0$** | `Status_Udara` independen (tidak berkaitan) dengan `Jam_Sibuk` |
| **$H_1$** | `Status_Udara` dependen (berkaitan) dengan `Jam_Sibuk` |
| **Statistik Uji** | $\chi^2 = {chi2:.3f}$ ($df = {dof}$, Cramér's $V = {cramers_v:.3f}$) |
| **$p$-value** | ${p_chi:.4g}$ |
| **Keputusan ($\alpha = 0,05$)** | **{keputusan_chi}** |
| **Interpretasi Bahasa Sehari-hari** | {interp_chi} |
"""))

In [ ]:
# OPSIONAL: Analisis tambahan Status_Udara x Zona_Lokasi sebagai pembanding spasial
obs2 = pd.crosstab(df["Zona_Lokasi"].astype(str).str.strip(), df["Status_Udara"].astype(str).str.strip())
if set(URUTAN_STATUS).issubset(set(obs2.columns)) and set(URUTAN_ZONA).issubset(set(obs2.index)):
    obs2 = obs2[URUTAN_STATUS].reindex(URUTAN_ZONA)
chi2_b, p_b, dof_b, exp_b = stats.chi2_contingency(obs2)
print(f"Pembanding Status_Udara x Zona_Lokasi: chi2 = {chi2_b:.2f}, df = {dof_b}, p = {p_b:.4g}")
print("Catatan: ada sel ekspektasi < 5?", bool((exp_b < 5).any()))
obs2

## Ringkasan Seluruh Uji & Bahan Laporan Minggu 3 (Untuk Technical Writer & Project Lead)

In [ ]:
ringkas = pd.DataFrame([
    ["Two-Sample Independent t-Test (Hari Kerja vs Akhir Pekan)", f"t = {t_stat:.3f} (df={df_t:.1f})", p_t, keputusan_t],
    ["One-Way ANOVA (3 Zona Lokasi)", f"F({df_antar},{df_dalam}) = {F:.3f}", p_f, keputusan_anova],
    ["Chi-Square Independence (Status_Udara x Jam_Sibuk)", f"chi2 = {chi2:.3f} (df={dof})", p_chi, keputusan_chi],
], columns=["Uji_Hipotesis", "Statistik_Uji", "p_value", "Keputusan_alpha_0.05"])
ringkas["p_value_fmt"] = ringkas["p_value"].map(lambda v: f"{v:.4g}")
simpan_tabel(ringkas, "03_ringkasan_uji_hipotesis", index=False)
display(ringkas)

# Cetak ringkasan teks siap tempel ke reports/draft/03_interpretasi_hasil_per_uji.md
print("\n" + "="*75)
print("DRAFT SIAP SALIN UNTUK reports/draft/03_interpretasi_hasil_per_uji.md")
print("="*75)
print(f"1. CI 95% Rata-rata PM2.5 : [{bb:.2f} ; {ba:.2f}] (Mean = {xbar:.2f})")
print(f"   CI 95% Proporsi Tidak Sehat : [{wald[0]*100:.2f}% ; {wald[1]*100:.2f}%] (p_hat = {p_hat*100:.2f}%)")
print(f"2. Independent t-Test   : t = {t_stat:.3f}, df = {df_t:.1f}, p = {p_t:.4g} -> {keputusan_t}")
print(f"3. One-Way ANOVA        : F({df_antar},{df_dalam}) = {F:.3f}, p = {p_f:.4g} -> {keputusan_anova}")
print(f"4. Chi-Square Test      : chi2 = {chi2:.3f}, df = {dof}, p = {p_chi:.4g} -> {keputusan_chi}")

## Serahkan hasil
Jalankan sel di bawah -> unduh ZIP -> upload ke folder `outputs/` di GitHub repository kelompok (`Tur-ai/udara-mobilitas-statsprob`).

In [ ]:
unduh_hasil("hasil_02b_inference")

### Checklist Penyelesaian Tahap 3 (Minggu 3)
- [x] CI 95% rata-rata populasi `Indeks_PM25` dan CI 95% proporsi kategori `Tidak Sehat` beserta interpretasi (Checklist 3a)
- [x] Two-Sample Independent t-Test: $H_0$, $H_1$, statistik $t$, $p$-value, keputusan statistik, dan interpretasi bahasa sehari-hari (Checklist 3b-1)
- [x] One-Way ANOVA: $H_0$, $H_1$, statistik $F$, $p$-value, keputusan statistik, uji lanjut Post-Hoc Tukey HSD, dan interpretasi (Checklist 3b-2)
- [x] Chi-Square Test of Independence: tabel frekuensi observasi vs ekspektasi, statistik $\chi^2$, $p$-value, keputusan statistik, dan interpretasi (Checklist 3b-3)
- [x] Seluruh tabel dan visualisasi tersimpan di `outputs/tables/` dan `outputs/figures/`